##  Z Transform: Averaging Lowpass, Combs, And FIR Filter Design
### Chris Tralie

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
from scipy import signal

### Moving Average Lowpass Filters

Let's look at a more general version of the moving average.  Its impuse response is:

#### $h[0] = 1, h[1] = 1, h[2] = 1, h[3], ..., h[A] = 1$

#### $y[n] = x[n] + x[n-1] + x[n-2] + ... x[n - A]$


Plugging in a phasor $e^{i \omega}$, we get

### $y[n] = e^{i \omega n}(1 + e^{-i \omega} + e^{-i 2 \omega} + ... + e^{-i A \omega} ) $

### $y[n] = e^{i \omega n}(z^0 + z^1 + z^2 + ... + z^A ), z = e^{-i \omega} $

Let's focus on the z-transform of $h$ in the parentheses.  This is a <a href = "https://mathworld.wolfram.com/GeometricSeries.html">geometric series</a>, so we can rewrite it as a fraction instead of a sum:

## $\frac{1 - z^{A+1}}{1 - z}  $

## $\frac{1 - e^{-i \omega(A+1)}}{1 - e^{-i \omega}}  $

Now let's multiply by a fancy 1 to convert the numerator to a canonical phasor:

## $\frac{1 - e^{-i \omega(A+1)}}{1 - e^{-i \omega}} \frac{e^{i \omega(A+1)/2}} {e^{i \omega(A+1)/2}} = \frac{1}{e^{i \omega(A+1)/2}} \frac{e^{i \omega (A+1)/2} - e^{-i \omega(A+1)/2}}{1 - e^{-i \omega}}   $

Then we can apply Euler's formula to get:

### $\frac{1}{e^{i \omega(A+1)/2}} \frac{ 2 i \sin(\omega (A+1)/2)}{1 - e^{-i \omega}} $

Once again, we multiply by a fancy 1 to convert the denominator to a canonical phasor:

### $\frac{1}{e^{i \omega(A+1)/2}} \frac{ 2 i \sin(\omega (A+1)/2)}{1 - e^{-i \omega}} \frac{e^{i \omega/2}} {e^{i \omega/2}} = \frac{e^{i \omega/2}}{e^{i \omega(A+1)/2}} \frac{ 2 i \sin(\omega (A+1)/2)}{e^{i \omega/2} - e^{-i \omega/2}}  $

### $ \frac{e^{i \omega/2}}{e^{i \omega(A+1)/2}} \frac{ \sin(\omega (A+1)/2)}{  \sin(\omega/2) } $

In [ ]:
def get_moving_avg_mag(ws, A):
    return np.abs(np.sin(ws*(A+1)/2)/np.sin(ws/2))

ws = np.linspace(0, np.pi, 100)
As = [1, 5, 10]
for A in As:
    plt.plot(ws, get_moving_avg_mag(ws, A))
plt.legend(As)

### Comb Filter

Some spacing $T$

### $h[0] = 1, h[T], h[2T] = 1, ..., h[AT] = 1$


In [ ]:
sr = 44100
T = 100
A = 20
h = np.zeros(A*T+1)
h[0::T] = 1
x = np.random.randn(sr*4)
y = np.convolve(x, h)
ipd.Audio(y, rate=sr)

Magnitude of z-transform of a comb filter with spacing $T$ between the teeth and $A+1$ teeth total

### $| \frac{ \sin(\omega T (A+1)/2)}{  \sin(T \omega/2) } |$

Below is a plot of an example numerically.  Increasing $A$ makes the peaks more pronounced, while increasing $T$ moves the frequency of the peaks down (as we would expect from a comb filter)


In [ ]:
def get_comb_mag(ws, A, T):
    return np.abs(np.sin(ws*T*(A+1)/2)/np.sin(T*ws/2))

ws = np.linspace(0, np.pi, 1000)
A = 20
T = 10

plt.figure(figsize=(6, 6))
plt.subplot(211)
h = np.zeros(A*T+1)
h[0::T] = 1
plt.stem(h)
plt.subplot(212)
plt.plot(ws/(2*np.pi), get_comb_mag(ws, A, T))
plt.xlabel("Cycles per sample")
plt.title(f"z-Transform Magnitude of Comb Filter with A={A}, T={T}")
plt.tight_layout()

## Filter Design

We can use the Discrete Fourier Transform to design filters in the frequency domain, which is a much more intuitive way to specify cutoff frequencies for lowpass, bandpass, and highpass filters.  The filter is then the inverse DFT of the frequency specification.  

But a direct inverse can lead towards a lot of ripples because of the windowing effect.  To mitigate this, we can put a Hann window on the inverse before using it.

We might still struggle with too gradual of a frequency cutoff though.  The way to make the frequency cutoff sharper is to increase the number of samples in the filter.  We then lose some time resolution, which could be a downside if frequencies are changing rapidly in our signal.  But this is one of the limitations of using finite impulse response filters

In [ ]:
hann_window = lambda N: (0.5*(1 - np.cos(2*np.pi*np.arange(N)/N))).astype(np.float32)

def z_transform(h, z):
    ret = 0
    for n in range(len(h)):
        ret += h[n]*(z**(-n))
    return ret

def get_z_over_unit_semicircle(h, n_samples=10000):
    ws = np.linspace(0, np.pi, n_samples)
    ret = np.zeros(n_samples, dtype=complex)
    for i, w in enumerate(ws):
        ret[i] = z_transform(h, np.exp(1j*w))
    return ws, ret


N = 32 # Window length
fc = 1/8 # cycles / sample
k = int(fc*N) # (cycles/sample) * (samples / window) = cycles/windows
H = np.zeros(N//2+1)
H[0:k+1] = 1   
h = np.fft.irfft(H)
h = np.fft.fftshift(h)
print(len(h))

plt.figure(figsize=(12, 6))
plt.subplot(231)
plt.title("Direct Inverse FFT")
plt.plot(h)
plt.subplot(234)
ws, ret = get_z_over_unit_semicircle(h, 10000)
plt.plot(ws/(2*np.pi), np.abs(ret))
plt.axvline(fc, linestyle='--')
plt.xlabel("Cycles/sample")
plt.title("Z-Transform")

plt.subplot(232)
plt.title("Windowed Inverse FFT")
h = h*hann_window(h.size)
plt.plot(h)
plt.plot(hann_window(h.size)*np.max(h), linestyle='--')
plt.subplot(235)
ws, ret = get_z_over_unit_semicircle(h, 10000)
plt.plot(ws/(2*np.pi), np.abs(ret))
plt.axvline(fc, linestyle='--')
plt.xlabel("Cycles/sample")
plt.title("Z-Transform")


N = 1024 # Window length
fc = 1/8 # cycles / sample
k = int(fc*N) # (cycles/sample) * (samples / window) = cycles/windows
H = np.zeros(N//2+1)
H[0:k+1] = 1   
h = np.fft.irfft(H)
h = np.fft.fftshift(h)
print(len(h))
plt.subplot(233)
plt.title("Windowed 1024 Inverse FFT")
h = h*hann_window(h.size)
plt.plot(h)
plt.plot(hann_window(h.size)*np.max(h), linestyle='--')
plt.subplot(236)
ws, ret = get_z_over_unit_semicircle(h, 10000)
plt.plot(ws/(2*np.pi), np.abs(ret))
plt.axvline(fc, linestyle='--')
plt.xlabel("Cycles/sample")
plt.title("Z-Transform")
plt.tight_layout()

If we apply this strategy to design a finite impulse lowpass filter for audio but we don't use the Hann window, there is audible "ringing" in the result.  We'll show this by filtering a linear chirp that starts at 50hz and goes up at a rate of 500hz per second

In [ ]:
sr = 44100
# Chirp df = 50 + 500*t,  f = 50*t + 250*t^2
t = np.arange(sr*3)/sr
f = 50*t + 250*t**2
x = np.cos(2*np.pi*f)
x = np.concatenate((x, x[::-1]))
ipd.Audio(x, rate=sr)

Here is the filter design and result.  It succeeded as a lowpass filter, but there is audible ringing

In [ ]:
H = np.zeros(sr//2+1) # The kth bin goes through k cycles/sec
H[0:550] = 1
h = np.fft.irfft(H)
h = np.fft.fftshift(h)
y = signal.fftconvolve(x, h)
ipd.Audio(y, rate=sr)

The ringing is cut down as soon as we use the Hann window

In [ ]:
H = np.zeros(sr//2+1) # The kth bin goes through k cycles/sec
H[0:550] = 1
h = np.fft.irfft(H)
h = np.fft.fftshift(h)
h = h*hann_window(h.size)
y = signal.fftconvolve(x, h)
ipd.Audio(y, rate=sr)

Finally, for one more example, here is a bandpass filter

In [ ]:
H = np.zeros(sr//2+1) # The kth bin goes through k cycles/sec
H[550:1000] = 1
h = np.fft.irfft(H)
h = np.fft.fftshift(h)
h = h*hann_window(h.size)
y = signal.fftconvolve(x, h)
plt.plot(h)
plt.xlim([20000, 25000])
ipd.Audio(y, rate=sr)